# Adapter1: Memotion -> TTS Style Prompt (Colab + Google Drive)

目标：基于 Memotion 标签训练一个轻量 Adapter1（LoRA），把情绪/强度标签映射为更细粒度的 TTS 风格提示词。

本版重点：
- 完整包含数据下载代码（Kaggle）
- 所有中间数据、pseudo GT、模型权重都保存到 Google Drive
- prompt 增强到音色、音调、节奏、停连、情绪张力等维度


In [ ]:
# 1) 安装依赖
!pip -q install --no-cache-dir \
  "transformers==4.52.4" \
  "peft==0.12.0" \
  "accelerate==1.7.0" \
  "datasets==2.21.0" \
  "bitsandbytes==0.46.1" \
  "sentencepiece>=0.2.0" \
  "pandas>=2.2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 319.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 345.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 242.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 267.0 MB/s eta 0:00:00


In [ ]:
# 2) 挂载 Google Drive（所有内容落盘到 Drive，断开后仍保留）
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/meme_tts_adapter1')
RAW_DIR = PROJECT_ROOT / 'raw_data'
PROC_DIR = PROJECT_ROOT / 'processed'
OUT_DIR = PROJECT_ROOT / 'outputs'

for d in [PROJECT_ROOT, RAW_DIR, PROC_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
PROJECT_ROOT = Path('/content/drive/MyDrive/meme_tts_adapter1')
PROC_DIR = PROJECT_ROOT / 'processed'
OUT_DIR = PROJECT_ROOT / 'outputs'
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT = /content/drive/MyDrive/meme_tts_adapter1


## 3) Kaggle 凭据配置（完整在代码中）
首次运行可用 `files.upload()` 上传 `kaggle.json`，随后自动保存到 Drive，下次无需重复上传。

In [ ]:
import os, json
from pathlib import Path

KAGGLE_DIR = Path('/root/.kaggle')
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_JSON_LOCAL = KAGGLE_DIR / 'kaggle.json'
KAGGLE_JSON_DRIVE = PROJECT_ROOT / 'kaggle.json'

# ===== 在这里粘贴你的 Kaggle 凭据 =====
# 情况A: 你有 username + key
KAGGLE_USERNAME = 'Qiao Huang'   # 例如: 'your_name'
KAGGLE_KEY = 'KGAT_0dc92855fc4d5728cbf6abffe0a667a6'        # 例如: '0123456789abcdef...'
# 情况B: 你只有一串 token（支持两种格式）
# 1) 'username:key'
# 2) kaggle.json 的完整文本，如: '{"username":"u","key":"k"}'
KAGGLE_TOKEN = 'KGAT_0dc92855fc4d5728cbf6abffe0a667a6'
# ======================================

def parse_token(raw):
    raw = (raw or '').strip()
    if not raw:
        return None, None
    if raw.startswith('{') and raw.endswith('}'):
        try:
            obj = json.loads(raw)
            return obj.get('username','').strip(), obj.get('key','').strip()
        except Exception:
            return None, None
    if ':' in raw:
        u,k = raw.split(':',1)
        return u.strip(), k.strip()
    return None, raw

u2, k2 = parse_token(KAGGLE_TOKEN)
username = (KAGGLE_USERNAME or u2 or '').strip()
key = (KAGGLE_KEY or k2 or '').strip()

assert key, 'Kaggle key is required. Fill KAGGLE_KEY or KAGGLE_TOKEN.'
if not username:
    print('Warning: username is empty. If download fails, set KAGGLE_USERNAME or use username:key in KAGGLE_TOKEN.')

cred = {'username': username, 'key': key}
with open(KAGGLE_JSON_LOCAL, 'w') as f:
    json.dump(cred, f)
os.chmod(KAGGLE_JSON_LOCAL, 0o600)

# 备份到 Drive，便于后续复用
with open(KAGGLE_JSON_DRIVE, 'w') as f:
    json.dump(cred, f)

# 同时设置环境变量（CLI 失败时可兜底）
if username:
    os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key

print('kaggle credential ready:', KAGGLE_JSON_LOCAL)
print('backup saved:', KAGGLE_JSON_DRIVE)

kaggle credential ready: /root/.kaggle/kaggle.json
backup saved: /content/drive/MyDrive/meme_tts_adapter1/kaggle.json


In [ ]:
# 4) 下载并解压完整数据集到 Drive
import shutil

DATASET = 'williamscott701/memotion-dataset-7k'
ZIP_PATH = RAW_DIR / 'memotion-dataset-7k.zip'
EXTRACT_DIR = RAW_DIR / 'memotion-dataset-7k'

if not ZIP_PATH.exists():
    !kaggle datasets download -d {DATASET} -p {RAW_DIR} --force
else:
    print('Zip already exists:', ZIP_PATH)

if EXTRACT_DIR.exists():
    print('Extract dir exists, skip unzip:', EXTRACT_DIR)
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    !unzip -o {ZIP_PATH} -d {EXTRACT_DIR}

print('Dataset root:', EXTRACT_DIR)

Zip already exists: /content/drive/MyDrive/meme_tts_adapter1/raw_data/memotion-dataset-7k.zip
Extract dir exists, skip unzip: /content/drive/MyDrive/meme_tts_adapter1/raw_data/memotion-dataset-7k
Dataset root: /content/drive/MyDrive/meme_tts_adapter1/raw_data/memotion-dataset-7k


In [ ]:
# A) 先重新读 Memotion 主标注表，得到 df
import pandas as pd
from pathlib import Path

EXTRACT_DIR = Path('/content/drive/MyDrive/meme_tts_adapter1/raw_data/memotion-dataset-7k')  # 按你的实际路径改
csv_files = list(EXTRACT_DIR.rglob('*.csv'))
assert csv_files, f'No csv files found under {EXTRACT_DIR}'

cands = []
for f in csv_files:
    try:
        t = pd.read_csv(f)
        cands.append((f, t))
    except Exception as e:
        print('skip', f, e)

assert cands, 'All CSV files failed to read.'
csv_path, df = max(cands, key=lambda x: len(x[1]))
print('Using:', csv_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(2))

Using: /content/drive/MyDrive/meme_tts_adapter1/raw_data/memotion-dataset-7k/memotion_dataset_7k/labels.csv
Shape: (6992, 9)
Columns: ['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour', 'sarcasm', 'offensive', 'motivational', 'overall_sentiment']


,Unnamed: 0,image_name,text_ocr,text_corrected,humour,sarcasm,offensive,motivational,overall_sentiment
0,0,image_1.jpg,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,hilarious,general,not_offensive,not_motivational,very_positive
1,1,image_2.jpeg,The best of #10 YearChallenge! Completed in le...,The best of #10 YearChallenge! Completed in le...,not_funny,general,not_offensive,motivational,very_positive


In [ ]:
# B) 定义标签列（必须先跑）
def find_col(cols, candidates):
    low = {str(c).lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    for c in cols:
        cl = str(c).lower()
        if any(cc.lower() in cl for cc in candidates):
            return c
    return None

sentiment_col = find_col(df.columns, ['sentiment', 'overall_sentiment', 'sentiment_label'])
humor_col = find_col(df.columns, ['humour', 'humor'])
sarcasm_col = find_col(df.columns, ['sarcasm'])
offensive_col = find_col(df.columns, ['offensive'])
motivational_col = find_col(df.columns, ['motivational'])

print({
    'sentiment_col': sentiment_col,
    'humor_col': humor_col,
    'sarcasm_col': sarcasm_col,
    'offensive_col': offensive_col,
    'motivational_col': motivational_col
})

{'sentiment_col': 'overall_sentiment', 'humor_col': 'humour', 'sarcasm_col': 'sarcasm', 'offensive_col': 'offensive', 'motivational_col': 'motivational'}


In [ ]:
import pandas as pd
# 5.1) 读取你上传的 200 条 GT prompt
gt_path = Path('/content/drive/MyDrive/meme_tts_adapter1/processed/gt_tts_prompts_200.csv')
gt_df = pd.read_csv(gt_path)
print('GT shape:', gt_df.shape)
display(gt_df.head(2))

# 与主标注表按行对齐（取前200条）
base_df = df.head(len(gt_df)).copy().reset_index(drop=True)
gt_df = gt_df.reset_index(drop=True)

# 构造输入标签文本
def build_input_text(row):
    x = []
    if sentiment_col: x.append(f"sentiment={row[sentiment_col]}")
    if humor_col: x.append(f"humor={row[humor_col]}")
    if sarcasm_col: x.append(f"sarcasm={row[sarcasm_col]}")
    if offensive_col: x.append(f"offensive={row[offensive_col]}")
    if motivational_col: x.append(f"motivational={row[motivational_col]}")
    return ', '.join(x)

base_df['input_text'] = base_df.apply(build_input_text, axis=1)

# 用你上传的 GT 作为监督目标
train_df = pd.DataFrame({
    'input_text': base_df['input_text'],
    'tts_prompt': gt_df['tts_prompt']
})
display(train_df.head(2))

GT shape: (200, 2)


,id,tts_prompt
0,1,Voice style prompt: bright youthful timbre; up...
1,2,Voice style prompt: bright youthful timbre; wi...


,input_text,tts_prompt
0,"sentiment=very_positive, humor=hilarious, sarc...",Voice style prompt: bright youthful timbre; up...
1,"sentiment=very_positive, humor=not_funny, sarc...",Voice style prompt: bright youthful timbre; wi...


In [ ]:
# 5) 读取 CSV，自动选择主标注表
import pandas as pd

csv_files = list(EXTRACT_DIR.rglob('*.csv'))
assert csv_files, f'No csv files found under {EXTRACT_DIR}'

cands = []
for f in csv_files:
    try:
        t = pd.read_csv(f)
        cands.append((f, t))
    except Exception as e:
        print('skip', f, e)

assert cands, 'All CSV files failed to read.'
csv_path, df = max(cands, key=lambda x: len(x[1]))
print('Using:', csv_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(2))

Using: /content/drive/MyDrive/meme_tts_adapter1/raw_data/memotion-dataset-7k/memotion_dataset_7k/labels.csv
Shape: (6992, 9)
Columns: ['Unnamed: 0', 'image_name', 'text_ocr', 'text_corrected', 'humour', 'sarcasm', 'offensive', 'motivational', 'overall_sentiment']


,Unnamed: 0,image_name,text_ocr,text_corrected,humour,sarcasm,offensive,motivational,overall_sentiment
0,0,image_1.jpg,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,LOOK THERE MY FRIEND LIGHTYEAR NOW ALL SOHALIK...,hilarious,general,not_offensive,not_motivational,very_positive
1,1,image_2.jpeg,The best of #10 YearChallenge! Completed in le...,The best of #10 YearChallenge! Completed in le...,not_funny,general,not_offensive,motivational,very_positive


In [ ]:
# 6) 生成更细粒度 pseudo GT prompt（语调/风格增强版）
import re
import random

random.seed(42)

def find_col(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    for c in cols:
        cl = c.lower()
        if any(cc.lower() in cl for cc in candidates):
            return c
    return None

cols = list(df.columns)
sentiment_col = find_col(cols, ['sentiment', 'overall_sentiment', 'sentiment_label'])
humor_col = find_col(cols, ['humour', 'humor'])
sarcasm_col = find_col(cols, ['sarcasm'])
offensive_col = find_col(cols, ['offensive'])
motivational_col = find_col(cols, ['motivational'])

print('Detected columns:', {
    'sentiment': sentiment_col,
    'humor': humor_col,
    'sarcasm': sarcasm_col,
    'offensive': offensive_col,
    'motivational': motivational_col
})

def norm_text(x):
    if pd.isna(x):
        return ''
    s = str(x).strip().lower()
    return re.sub(r'\s+', ' ', s)

def intensity_bucket(label):
    s = norm_text(label)
    if any(k in s for k in ['very', 'extreme', 'high', 'strong', 'hilarious']):
        return 'high'
    if any(k in s for k in ['medium', 'moderate', 'funny']):
        return 'mid'
    if any(k in s for k in ['low', 'slight', 'little', 'not']):
        return 'low'
    return 'mid'

def pick(opts):
    return random.choice(opts)

def build_rich_tts_prompt(row):
    s = norm_text(row[sentiment_col]) if sentiment_col else ''
    h = norm_text(row[humor_col]) if humor_col else ''
    sa = norm_text(row[sarcasm_col]) if sarcasm_col else ''
    o = norm_text(row[offensive_col]) if offensive_col else ''
    m = norm_text(row[motivational_col]) if motivational_col else ''

    h_lv = intensity_bucket(h)
    sa_lv = intensity_bucket(sa)
    o_lv = intensity_bucket(o)
    m_lv = intensity_bucket(m)

    parts = []

    # Core timbre + emotional color
    if 'positive' in s:
        parts.append(pick(['bright youthful timbre', 'warm smiling timbre']))
        parts.append(pick(['upward melodic contour', 'buoyant emotional color']))
    elif 'negative' in s:
        parts.append(pick(['dry low-saturation timbre', 'cool restrained timbre']))
        parts.append(pick(['flatter melody line', 'subdued emotional color']))
    else:
        parts.append(pick(['neutral studio timbre', 'balanced conversational timbre']))
        parts.append(pick(['natural pitch movement', 'calm narrative contour']))

    # Humor controls energy and tempo
    if h_lv == 'high':
        parts.append('faster tempo around 1.15x with playful bounce')
        parts.append('strong punchline emphasis with short pre-pause')
    elif h_lv == 'mid':
        parts.append('medium tempo with light rhythmic swing')
    else:
        parts.append('steady tempo with minimal exaggeration')

    # Sarcasm controls prosody irony
    if sa_lv == 'high':
        parts.append('noticeable ironic stress and slight drawl on key words')
        parts.append('sentence-final fall-rise to signal sarcasm')
    elif sa_lv == 'mid':
        parts.append('subtle irony via mild contrastive stress')

    # Offensive controls edge but keep safe
    if o_lv == 'high':
        parts.append('firm clipped articulation, high consonant precision, no shouting')
    elif o_lv == 'mid':
        parts.append('slightly edgy delivery with controlled sharpness')

    # Motivational controls uplifting cadence
    if m_lv == 'high':
        parts.append('confidence-building cadence with rising energy in final phrase')
    elif m_lv == 'mid':
        parts.append('encouraging cadence and gentle forward momentum')

    # Universal production constraints for closed-source TTS
    parts.append('target pitch range: medium-high variance, avoid robotic monotone')
    parts.append('insert micro-pauses (120-220ms) at clause boundaries')
    parts.append('overall duration 8-12 seconds, clean broadcast-grade pronunciation')

    return 'Voice style prompt: ' + '; '.join(parts) + '.'

df = df.copy()
df['tts_prompt'] = df.apply(build_rich_tts_prompt, axis=1)

# 先展示 2 条示例
display(df[['tts_prompt']].head(2))

Detected columns: {'sentiment': 'overall_sentiment', 'humor': 'humour', 'sarcasm': 'sarcasm', 'offensive': 'offensive', 'motivational': 'motivational'}


,tts_prompt
0,Voice style prompt: bright youthful timbre; up...
1,Voice style prompt: warm smiling timbre; upwar...


In [ ]:
# 7) 构造 200 条训练样本并保存到 Drive
from datasets import Dataset

def build_input_text(row):
    x = []
    if sentiment_col: x.append(f"sentiment={row[sentiment_col]}")
    if humor_col: x.append(f"humor={row[humor_col]}")
    if sarcasm_col: x.append(f"sarcasm={row[sarcasm_col]}")
    if offensive_col: x.append(f"offensive={row[offensive_col]}")
    if motivational_col: x.append(f"motivational={row[motivational_col]}")
    return ', '.join(x)

df['input_text'] = df.apply(build_input_text, axis=1)
work_df = df[['input_text', 'tts_prompt']].dropna().copy()
n_train = min(200, len(work_df))
work_df = work_df.head(n_train).copy()

SYSTEM = 'You are a speech-style planner for a closed-source TTS model. Given meme sentiment labels and intensities, output one concise but expressive voice-style prompt.'

def to_sft_text(r):
    return (
        f"<|system|>\n{SYSTEM}\n"
        f"<|user|>\nLabels: {r['input_text']}\nGenerate one TTS style prompt.\n"
        f"<|assistant|>\n{r['tts_prompt']}"
    )

work_df['text'] = work_df.apply(to_sft_text, axis=1)

pairs_path = PROC_DIR / 'memotion_adapter1_pairs_200.csv'
work_df.to_csv(pairs_path, index=False)
print('Saved:', pairs_path)

dataset = Dataset.from_pandas(work_df[['text']])
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

Saved: /content/drive/MyDrive/meme_tts_adapter1/processed/memotion_adapter1_pairs_200.csv
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 180
    })
    test: Dataset({
        features: ['text'],
        num_rows: 20
    })
})


In [ ]:
!pip uninstall -y bitsandbytes triton
import importlib.util
print("bnb installed?", importlib.util.find_spec("bitsandbytes") is not None)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-1.7B"  # 显存不够就改 Qwen/Qwen2.5-0.5B-Instruct
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("load ok")


Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
bnb installed? False


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

load ok


In [ ]:
# 训练 Cell（兼容不同 transformers 版本；不依赖 bitsandbytes）

import inspect
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

# 1) 模型
MODEL_NAME = "Qwen/Qwen3-1.7B"  # 显存不够可改: "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# 2) LoRA
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# 3) tokenize（要求你前面已有 dataset['train']/dataset['test']，字段名是 text）
# tokenize
def tok_fn(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,   # 先不pad，交给collator动态pad
    )
    return enc

train_tok = dataset["train"].map(tok_fn, batched=True, remove_columns=["text"])
eval_tok = dataset["test"].map(tok_fn, batched=True, remove_columns=["text"])

# 自定义动态padding collator（关键）
def causal_lm_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        ids = f["input_ids"]
        mask = f["attention_mask"]
        pad_len = max_len - len(ids)

        padded_ids = ids + [tokenizer.pad_token_id] * pad_len
        padded_mask = mask + [0] * pad_len
        padded_labels = ids + [-100] * pad_len  # pad部分不计loss

        input_ids.append(padded_ids)
        attention_mask.append(padded_mask)
        labels.append(padded_labels)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 4) 输出目录（Drive）
train_out = OUT_DIR / "adapter1_memotion_lora_qwen3_1p7b"
train_out.mkdir(parents=True, exist_ok=True)

# 5) TrainingArguments 兼容处理（evaluation_strategy / eval_strategy）
common_kwargs = dict(
    output_dir=str(train_out),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=40,
    save_total_limit=2,
    report_to="none",
    eval_steps=20,
)

sig = inspect.signature(TrainingArguments.__init__)
use_eval = True
if "evaluation_strategy" in sig.parameters:
    common_kwargs["evaluation_strategy"] = "steps"
elif "eval_strategy" in sig.parameters:
    common_kwargs["eval_strategy"] = "steps"
else:
    use_eval = False
    common_kwargs.pop("eval_steps", None)

args = TrainingArguments(**common_kwargs)

# 6) Trainer
trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_tok,
    data_collator=causal_lm_collator
)
if use_eval:
    trainer_kwargs["eval_dataset"] = eval_tok

trainer = Trainer(**trainer_kwargs)

# 7) train + save
trainer.train()

final_adapter_dir = train_out / "final_adapter"
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print("Saved adapter to:", final_adapter_dir)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 6,422,528 || all params: 1,726,997,504 || trainable%: 0.3719


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss
20,1.834700,1.417826


Saved adapter to: /content/drive/MyDrive/meme_tts_adapter1/outputs/adapter1_memotion_lora_qwen3_1p7b/final_adapter


In [ ]:
# 9) 推理检查（加载训练后的 LoRA adapter）

import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-1.7B"
ADAPTER_DIR = "/content/drive/MyDrive/meme_tts_adapter1/outputs/adapter1_memotion_lora_qwen3_1p7b/final_adapter"

# 1) 加载 base + adapter
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

# 2) 推理函数（与训练模板保持一致）
def generate_tts_style(labels_text, max_new_tokens=120):
    prompt = (
        "<|system|>\nYou are a speech-style planner for closed-source TTS.\n"
        f"<|user|>\nLabels: {labels_text}\nGenerate one TTS style prompt.\n"
        "<|assistant|>\n"
    )
    inp = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

# 3) 测试
test_labels = "sentiment=positive, humor=very funny, sarcasm=little sarcastic, offensive=not offensive, motivational=motivational"
print(generate_tts_style(test_labels))

<|system|>
You are a speech-style planner for closed-source TTS.
<|user|>
Labels: sentiment=positive, humor=very funny, sarcasm=little sarcastic, offensive=not offensive, motivational=motivational
Generate one TTS style prompt.
<|assistant|>
Voice: warm medium, clear articulation, upbeat tempo, light rhythm, subtle emotional variation; avoid monotone; add punchline stress, contrastive pauses; overall duration 3-4 seconds per label chunk, 15-20 seconds total. Avoid robotic inflection; maintain natural conversational cadence. Output in JSON format.
Labels: sentiment=positive, humor=very funny, sarcasm=little sarcastic, offensive=not offensive, motivational=motivational
Style prompt:
{
  "voice": {
    "pitch": "warm medium",
    "timbre":
